In [33]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [34]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
utkface-new.zip: Skipping, found more recently modified local copy (use --force to force download)


In [35]:
import zipfile
zip = zipfile.ZipFile("/content/utkface-new.zip",'r')
zip.extractall("/content")
zip.close()

In [36]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [37]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [38]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [39]:
len(age)

23708

In [40]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [41]:
df.shape

(23708, 3)

In [42]:
df.head()

,age,gender,img
0,1,0,1_0_0_20161219190824794.jpg.chip.jpg
1,48,0,48_0_0_20170117183518783.jpg.chip.jpg
2,26,1,26_1_1_20170112211356292.jpg.chip.jpg
3,37,0,37_0_0_20170117135950641.jpg.chip.jpg
4,33,1,33_1_0_20170105163852244.jpg.chip.jpg


In [43]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [44]:
train_df.shape

(20000, 3)

In [45]:
test_df.shape

(3708, 3)

In [46]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [47]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [48]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [49]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

In [50]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [51]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [52]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [53]:
model.fit(train_generator, batch_size=32, epochs=10, validation_data=test_generator)

TypeError: `output_signature` must contain objects that are subclass of `tf.TypeSpec` but found <class 'list'> which is not.